# AraBERT v02 reproduction — Colab workflow

This notebook checks the updated project, runs smoke checks in their separate smoke directory, and generates validation predictions from the existing seed 42 best checkpoint. Full training is disabled by default.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import os
import sys
from google.colab import drive

drive.mount('/content/drive')
project_root = Path('/content/drive/MyDrive/mizan-arabert-sentiment')
os.chdir(project_root)
sys.path.insert(0, str(project_root))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%pip install -q -r requirements-bonus.txt

In [4]:
import subprocess

git_dir = project_root / '.git'
print(f'.git directory exists: {git_dir.is_dir()}')
if git_dir.is_dir():
    commit_hash = subprocess.run(
        ['git', '-C', str(project_root), 'rev-parse', 'HEAD'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    print(f'Current commit: {commit_hash}')
else:
    print('Current commit: unavailable (.git directory not found)')

for split_name in ('train.csv', 'validation.csv'):
    split_path = project_root / 'data' / 'splits' / 'arsarcasm_v2' / split_name
    print(f'{split_path.relative_to(project_root)} exists: {split_path.is_file()}')

run_dir = project_root / 'checkpoints' / 'bonus' / 'repro_arabertv02_base'
print(f'\nSeed 42 run directory: {run_dir}')
if run_dir.is_dir():
    entries = sorted(path for path in run_dir.rglob('*') if path.is_file())
    if entries:
        for path in entries:
            print(f'{path.name}: {path.stat().st_size} bytes')
    else:
        print('(no files)')
else:
    print('(directory does not exist)')

try:
    from src.bonus.train import generate_validation_predictions_from_checkpoint
except (ImportError, AttributeError) as exc:
    raise RuntimeError(
        'The code on Drive is missing generate_validation_predictions_from_checkpoint. '
        'Please git pull or upload the updated files, then rerun this notebook.'
    ) from exc
print('Updated validation prediction function is available.')

.git directory exists: True
Current commit: aa17f21cf345c584cbcca55c6de56a4051472e94
data/splits/arsarcasm_v2/train.csv exists: True
data/splits/arsarcasm_v2/validation.csv exists: True

Seed 42 run directory: /content/drive/MyDrive/mizan-arabert-sentiment/checkpoints/bonus/repro_arabertv02_base
checkpoint_best.pt: 1622637913 bytes
checkpoint_last.pt: 1622637913 bytes
history.csv: 7371 bytes
checkpoint_best.pt: 1622627215 bytes
checkpoint_last.pt: 1622628559 bytes
config.json: 1375 bytes
history.csv: 1801 bytes
validation_predictions.csv: 1527 bytes
checkpoint_best.pt: 1622627215 bytes
checkpoint_last.pt: 1622627215 bytes
config.json: 1375 bytes
history.csv: 1237 bytes
validation_predictions.csv: 1527 bytes
checkpoint_best.pt: 1622627215 bytes
checkpoint_last.pt: 1622628495 bytes
config.json: 1388 bytes
history.csv: 1804 bytes
validation_predictions.csv: 1527 bytes
Updated validation prediction function is available.


## Smoke checks

These smoke checks write only under the configured smoke output directory. They do not touch or overwrite the seed 42 run directory. This is the first training-related step after setup and the environment check.

In [9]:
from src.bonus.train import run_notebook_workflow

smoke_results = run_notebook_workflow(smoke_mode=True)
display(smoke_results)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/content

epoch 1/2: train_loss=1.1496 validation_loss=1.0200 validation_macro_f_pn=0.2756


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/content

epoch 1/2: train_loss=1.1496 validation_loss=1.0200 validation_macro_f_pn=0.2756


... (3 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/content

epoch 2/2: train_loss=0.9797 validation_loss=1.0051 validation_macro_f_pn=0.2090


{'status': 'passed',
 'runtime_seconds': 1019.1891026109997,
 'training_samples': 200,
 'initial_epochs_per_run': 1,
 'resumed_epoch_count': 2,
 'runs': 2,
 'resume_history_epochs': [1, 2],
 'same_seed_histories_identical': True,
 'resume_epoch_two_matches_uninterrupted': True,
 'uninterrupted_history_epochs': [1, 2],
 'test_csv_reads': 0,
 'git_ignored_checkpoint': True,
 'hand_metrics': {'accuracy': 0.6,
  'weighted_precision': 0.7,
  'weighted_recall': 0.6,
  'weighted_f1': 0.6,
  'macro_f_pn': 0.5833333333333333,
  'per_class': {'NEG': {'precision': 0.5, 'recall': 0.5, 'f1': 0.5},
   'NEU': {'precision': 0.5, 'recall': 1.0, 'f1': 0.6666666666666666},
   'POS': {'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666}},
  'confusion_matrix': [[1, 1, 0], [0, 1, 0], [1, 0, 1]]},
 'sample_run_metrics': {'epoch': 1,
  'train_accuracy': 0.335,
  'train_weighted_precision': 0.3615878601794993,
  'train_weighted_recall': 0.335,
  'train_weighted_f1': 0.3424709250788924,
  'train_macro_f_

## Seed 42 validation predictions

This uses the existing `checkpoint_best.pt` and validation data only. It does not train or read the held-out test data.

In [5]:
import pandas as pd
from src.bonus.train import generate_validation_predictions_from_checkpoint

baseline_config = project_root / 'configs' / 'bonus' / 'repro_arabertv02_base.yaml'
best_checkpoint = run_dir / 'checkpoint_best.pt'
if not best_checkpoint.is_file():
    print(f'Skipping validation predictions: {best_checkpoint} does not exist.')
else:
    prediction_result = generate_validation_predictions_from_checkpoint(
        baseline_config, output_dir=run_dir
    )
    display(prediction_result['metrics'])
    predictions = pd.read_csv(prediction_result['predictions_csv'])
    display(predictions.head())

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'accuracy': 0.7434262948207171,
 'weighted_precision': 0.745635369601184,
 'weighted_recall': 0.7434262948207171,
 'weighted_f1': 0.7431515886776483,
 'macro_f_pn': 0.7108149933255367,
 'per_class': {'NEG': {'precision': 0.7318982387475538,
   'recall': 0.8095238095238095,
   'f1': 0.7687564234326825},
  'NEU': {'precision': 0.7912713472485768,
   'recall': 0.7252173913043478,
   'f1': 0.7568058076225045},
  'POS': {'precision': 0.6543778801843319,
   'recall': 0.6513761467889908,
   'f1': 0.6528735632183909}},
 'confusion_matrix': [[374, 65, 23], [106, 417, 52], [31, 45, 142]]}

,row_index,true_label,predicted_label
0,0,1,0
1,1,1,0
2,2,1,1
3,3,2,2
4,4,1,1


## Optional full training

Training is disabled by default. Set the switches below deliberately before running this section.

In [6]:
# Keep False to protect the already completed seed 42 run from retraining.
RUN_TRAINING = False
# True resumes a run; set False only to start a new, empty run directory.
RESUME = True

In [7]:
from src.bonus.train import run_training

baseline_config = project_root / 'configs' / 'bonus' / 'repro_arabertv02_base.yaml'
if not RUN_TRAINING:
    action = 'resume' if RESUME else 'start a new run'
    print(f'Training is disabled; would {action} at {run_dir} if RUN_TRAINING were True.')
else:
    existing_run_artifacts = [
        run_dir / 'checkpoint_best.pt',
        run_dir / 'history.csv',
    ]
    if not RESUME and any(path.exists() for path in existing_run_artifacts):
        raise RuntimeError(
            'Refusing to start a new run: checkpoint_best.pt or history.csv already '
            'exists in the output directory. Choose a new output directory or resume.'
        )
    if RESUME:
        # For seed 42, this would write config.json with the current commit, not the
        # commit that trained it. Resume is only for interrupted runs of seeds that
        # already have a matching config.json.
        training_history = run_training(
            baseline_config, output_dir=run_dir, resume=True
        )
    else:
        training_history = run_training(
            baseline_config, output_dir=run_dir, resume=False
        )
    display(training_history)

Training is disabled; would resume at /content/drive/MyDrive/mizan-arabert-sentiment/checkpoints/bonus/repro_arabertv02_base if RUN_TRAINING were True.


## History summary

Shows the requested per-epoch columns and reports which epoch has the highest validation macro F-PN.

In [8]:
import pandas as pd

history_path = run_dir / 'history.csv'
summary_columns = [
    'epoch',
    'validation_accuracy',
    'validation_weighted_f1',
    'validation_macro_f_pn',
    'validation_loss',
    'train_loss',
]
if not history_path.is_file():
    print(f'No history.csv found at {history_path}.')
else:
    history = pd.read_csv(history_path)
    history_summary = history[summary_columns].copy()
    best_row_index = history_summary['validation_macro_f_pn'].idxmax()
    best_epoch = int(history_summary.loc[best_row_index, 'epoch'])
    print(f'Highest validation_macro_f_pn: epoch {best_epoch}')
    display(history_summary)

Highest validation_macro_f_pn: epoch 10


,epoch,validation_accuracy,validation_weighted_f1,validation_macro_f_pn,validation_loss,train_loss
0,1,0.735458,0.735376,0.692195,0.619721,0.729499
1,2,0.743426,0.740375,0.684773,0.632942,0.557340
2,3,0.742629,0.741741,0.706246,0.598646,0.467722
3,4,0.749004,0.747739,0.699693,0.657055,0.382839
4,5,0.732271,0.729258,0.680226,0.734148,0.302421
5,6,0.743426,0.740895,0.689891,0.781028,0.232707
6,7,0.741036,0.739724,0.694256,0.871118,0.180232
7,8,0.737849,0.739339,0.696554,0.926028,0.144305
8,9,0.741036,0.740882,0.697895,0.993060,0.120064
9,10,0.743426,0.743152,0.710815,1.100060,0.095558
